In [0]:
# -----------------------------------------
# Healthcare Readmission Analytics Project
# Gold Layer Business Metrics
# -----------------------------------------

from pyspark.sql.functions import *

# Silver source table
silver_table = "workspace.silver.diabetic_data_silver"

# Gold target table
gold_table = "workspace.gold.healthcare_kpis_gold"

In [0]:
# Read Silver Layer

df_silver = spark.table(silver_table)

display(df_silver)

In [0]:
# Calculate readmission rate

readmission_rate_df = (
    df_silver
    .agg(
        round(
            avg("readmitted_flag") * 100,
            2
        ).alias("readmission_rate_percentage")
    )
)

display(readmission_rate_df)

In [0]:
# Average hospital stay

avg_stay_df = (
    df_silver
    .agg(
        round(
            avg("days_in_hospital"),
            2
        ).alias("average_hospital_stay")
    )
)

display(avg_stay_df)

In [0]:
# Patients by gender

gender_df = (
    df_silver
    .groupBy("gender")
    .count()
    .orderBy(desc("count"))
)

display(gender_df)

In [0]:
# Patients by race

race_df = (
    df_silver
    .groupBy("race")
    .count()
    .orderBy(desc("count"))
)

display(race_df)

In [0]:
# Medication usage distribution

medication_df = (
    df_silver
    .groupBy("medication_level")
    .count()
    .orderBy(desc("count"))
)

display(medication_df)

In [0]:
# Create aggregated healthcare metrics table

gold_kpi_df = (
    df_silver
    .groupBy(
        "age",
        "hospital_stay_category",
        "medication_level"
    )
    .agg(
        count("*").alias("total_patients"),
        
        round(
            avg("readmitted_flag") * 100,
            2
        ).alias("readmission_rate_percentage"),
        
        round(
            avg("days_in_hospital"),
            2
        ).alias("average_days_in_hospital"),
        
        round(
            avg("num_medications"),
            2
        ).alias("average_medications")
    )
)

In [0]:
# Preview Gold KPI dataframe

display(gold_kpi_df)

In [0]:
# Save Gold Layer table

(
    gold_kpi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

In [0]:
# Read Gold table

df_gold = spark.table(gold_table)

display(df_gold)

In [0]:
%sql
SELECT *
FROM workspace.gold.healthcare_kpis_gold
LIMIT 20;